In [6]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow.feather as feather
from scipy.stats import kstest

#import importlib
import sys
import os
hostname = os.uname().nodename
if hostname == 'BlackBeast':
    path = '/home/hedvigs/snake_book/econ'
    site = 'home'
elif hostname == 'hedvig-hp-elitedesk-800-g5-twr':
    path = '/home/hedvigs/PycharmProjects/homewrs/plab_workflow/'
    site = 'work'
elif hostname == 'work-computer':
    path = '/mnt/work/workbench/hedvigs/snake_book/econ'
    site = 'server'
elif hostname == 'wl-241113-007':
    path = '/home/hedvigs/wslGit/snake_book/econ'
    site = 'silverFlex'

sys.path.append(path)
print(path)
from workflow.scripts.data_management import setup_data as gt


/home/hedvigs/PycharmProjects/homewrs/plab_workflow/


In [7]:
def summarize_scores(
    subset=None, model=None, gen=None, fold=None, path=None, verbose=0
):
    inputs = [subset, gen, fold, model]
    config_names = ["subsets", "genomes", "folds", "models"]

    config_dict = {}

    for item, config_name in zip(inputs, config_names):
        if item is not None:
            config_dict[config_name] = [item]
        else:
            config_dict[config_name] = gt.read_config(config_name, path=path)
    y_pred = []
    missing = pd.DataFrame()
    k = 0

    for subset in config_dict["subsets"]:
        for model in config_dict["models"]:
            for gen in config_dict["genomes"]:
                for fold in config_dict["folds"]:
                    if subset == "all":
                        score_file = (
                            path
                            + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}_pruned.csv"
                        )
                    else:
                        score_file = (
                            path
                            + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}.csv"
                        )
                    try:
                        scores = pd.read_csv(score_file, sep="\t")
                        scores["fold"] = fold
                        scores["gen"] = gen
                        scores["model"] = model
                        scores["subset"] = subset
                        y_pred.append(scores)
                    except FileNotFoundError:
                        k += 1
                        missing.loc[k, "subset"] = subset
                        missing.loc[k, "model"] = model
                        missing.loc[k, "gen"] = gen
                        missing.loc[k, "fold"] = fold
    if k == 0:
        print(f" All files found!, {k} missing")
    elif verbose == 1:
        print(f"Files not found:", k)
    elif verbose == 2:
        print(f"Files not found:", k)
        subsetnames, subsetcounts = np.unique(missing["subset"], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f"  {sname}: {scount} missing")
            missubset = missing[missing["subset"] == sname]
            modelnames, modelcounts = np.unique(missubset["model"], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f"    {mname}: {mcount} missing")
    elif verbose == 3:
        print("Total files not found:", k)
        for colname in missing.columns:
            names, counts = np.unique(missing[colname], return_counts=True)
            print(f"{colname}: ")
            for name, count in zip(names, counts):
                print(f"  {name}: {count} missing")
    elif verbose == 4:
        print(f"Files not found:", k)
        subsetnames, subsetcounts = np.unique(missing["subset"], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f"  {sname}: {scount} missing")
            missubset = missing[missing["subset"] == sname]
            modelnames, modelcounts = np.unique(missubset["model"], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f"    {mname}: {mcount} missing")
                missmodel = missubset[missubset["model"] == mname]
                gennames, gencounts = np.unique(missmodel["gen"], return_counts=True)
                for gname, gcount in zip(gennames, gencounts):
                    print(f"        {gname}: {gcount} missing")

    full_set = pd.concat(y_pred)
    return full_set

In [8]:

    
subset = None #'top29' # wildcards["iSubset"]
model = None
gen = None  # wildcards["iGen"]

sum_df = summarize_scores(subset=subset, model=model, gen=gen, fold=None,path=path, verbose=4)
auc_df = sum_df.drop(columns=["number"]) # ,"fold", "train_score", "f1", "fbeta", "perm_score", "plr", "nlr", "pval_score", "auc_pred"])


Files not found: 169
  all: 65 missing
    bnb: 2 missing
        m: 2 missing
    knn: 13 missing
        combine: 4 missing
        f: 4 missing
        m: 5 missing
    lda: 7 missing
        combine: 2 missing
        f: 1 missing
        m: 4 missing
    lrc: 11 missing
        combine: 3 missing
        f: 3 missing
        m: 5 missing
    nn: 10 missing
        combine: 5 missing
        f: 2 missing
        m: 3 missing
    qda: 6 missing
        combine: 3 missing
        m: 3 missing
    rfc: 7 missing
        combine: 4 missing
        f: 1 missing
        m: 2 missing
    svc: 9 missing
        combine: 4 missing
        f: 1 missing
        m: 4 missing
  top5: 104 missing
    bnb: 11 missing
        combine: 4 missing
        f: 2 missing
        m: 5 missing
    knn: 13 missing
        combine: 5 missing
        f: 3 missing
        m: 5 missing
    lda: 12 missing
        combine: 5 missing
        f: 5 missing
        m: 2 missing
    lrc: 12 missing
        combine: 

In [9]:
auc_df


,test_perm,test_pval,test_score,train_score,auc_prob,auc_pred,r2,tn,fp,fn,tp,fold,gen,model,subset
0,0.49995,0.01796,0.53076,0.54170,0.53076,0.50808,-9.00068,1904,1272,49,35,2,f,bnb,top5
1,0.50057,0.01198,0.54094,0.55389,0.54094,0.52761,-9.12933,1839,1337,44,40,2,f,bnb,top5
2,0.49997,0.57086,0.49631,0.55571,0.49631,0.48456,-9.04925,1679,1497,47,37,2,f,bnb,top5
3,0.49954,0.75250,0.48834,0.54014,0.48834,0.47747,-9.00703,1823,1353,52,32,2,f,bnb,top5
4,0.49955,0.20958,0.51370,0.52145,0.51370,0.47741,-9.00923,1747,1429,50,34,2,f,bnb,top5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15,0.49971,0.00200,0.54654,1.00000,0.54930,0.50438,-8.94786,1578,1598,41,43,1,combine,svc,all
16,0.49869,0.00399,0.54963,0.00000,0.46684,0.52261,-8.98731,1656,1520,40,44,1,combine,svc,all
17,0.50048,0.00200,0.55320,1.00000,0.55309,0.52526,-8.97022,1635,1541,39,45,1,combine,svc,all
18,0.49983,0.95808,0.47075,0.50000,0.50000,0.54028,-8.95899,1617,1559,36,48,1,combine,svc,all


In [10]:

#sum_file= path + 'results/report/sum_file_top29.csv'
sum_file= path + f'results/report/sum_file_26.csv'
auc_df.to_csv(sum_file, sep="\t", index=False)
